In [1]:
from keras.layers import InputLayer
from keras.models import Sequential
from keras.layers import TimeDistributed
from keras.layers.convolutional import Conv2D
from keras.layers import Dense
from keras.layers.convolutional import MaxPooling2D
from keras.layers import Flatten
from keras.layers import LSTM
from keras.layers import Dense
from keras.models import load_model
from keras.models import model_from_json

import time
import datetime
from datetime import datetime
from keras.layers import Activation, Dense,Dropout
from keras.layers import BatchNormalization
from keras.layers import GlobalAveragePooling2D
import keras
from tensorflow.keras.optimizers import SGD



In [2]:
# Load the TensorBoard notebook extension
%load_ext tensorboard

In [3]:
batch_size = 16
frame_sequence = 16

In [4]:
model = Sequential()
#input layer
model.add(
    TimeDistributed(
        Conv2D(64,(3,3),padding='same', strides=(2,2), activation='relu'),
      input_shape=(frame_sequence, 224, 224, 3)
    )
)

# first conv, 64
model.add(
    TimeDistributed( 
        Conv2D(64, (3,3), 
            padding='same', strides=(2,2), activation='relu')
    )
)

#pooling
model.add(
    TimeDistributed(
        MaxPooling2D((2,2), strides=(2,2))
    )
)

# Second conv, 128
model.add(
    TimeDistributed(
        Conv2D(128, (3,3),
            padding='same', strides=(2,2), activation='relu')
    )
)

 
#pooling
model.add(
    TimeDistributed(
        MaxPooling2D((2,2), strides=(2,2))
    )
)

model.add(TimeDistributed(Flatten()))


model.add(
    LSTM(512, activation='relu', return_sequences=False)
)
model.add(Dropout(0.5))
model.add(Dense(256, activation='relu'))
model.add(Dropout(.3))
model.add(Dense(5, activation='softmax'))

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 time_distributed (TimeDistr  (None, 16, 112, 112, 64)  1792     
 ibuted)                                                         
                                                                 
 time_distributed_1 (TimeDis  (None, 16, 56, 56, 64)   36928     
 tributed)                                                       
                                                                 
 time_distributed_2 (TimeDis  (None, 16, 28, 28, 64)   0         
 tributed)                                                       
                                                                 
 time_distributed_3 (TimeDis  (None, 16, 14, 14, 128)  73856     
 tributed)                                                       
                                                                 
 time_distributed_4 (TimeDis  (None, 16, 7, 7, 128)    0

In [5]:
import os
os.listdir()

['.ipynb_checkpoints',
 'best_model1.h5',
 'best_model1.weights.h5',
 'cell.txt',
 'Dataset',
 'environment.yml',
 'fix_ipynb.py',
 'images',
 'LICENSE.txt',
 'logs',
 'model.json',
 'opt_cell.txt',
 'out.log',
 'patch_ipynb.py',
 'README.md',
 'run_training.py',
 'saved_models',
 'testing_model.ipynb',
 'TimeDistributedImageDataGenerator.py',
 'train.ipynb',
 '__pycache__']

In [6]:
from TimeDistributedImageDataGenerator import TimeDistributedImageDataGenerator

datagen = TimeDistributedImageDataGenerator(time_steps=16)

In [7]:
train_dir = r"Dataset/train_set"
val_dir = r"Dataset/val_set"



train_it = datagen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    batch_size=8,
    class_mode="categorical"
)

val_it = datagen.flow_from_directory(
    val_dir,
    target_size=(224,224),
    batch_size=8,
    class_mode="categorical"
)

Found 34221 images belonging to 5 classes.
Found 8552 images belonging to 5 classes.


In [8]:
# to find the batch and dimension of the loading images
batchX, batchy = train_it[0]

print('Batch shape=%s, min=%.3f, max=%.3f' % (batchX.shape, batchX.min(), batchX.max()))

Batch shape=(8, 16, 224, 224, 3), min=0.000, max=1.000


In [9]:
import os

# dataset path
dataset_path = "Dataset/train_set/"

# get class names
class_names = os.listdir(dataset_path)

# create class index dictionary
class_indices = {name: i for i, name in enumerate(class_names)}

print("Class Names:", class_names)
print("Class Indices:", class_indices)

Class Names: ['blast', 'blood', 'fight', 'gunshot', 'normal']
Class Indices: {'blast': 0, 'blood': 1, 'fight': 2, 'gunshot': 3, 'normal': 4}


In [10]:
import tensorflow
opt = tensorflow.keras.optimizers.SGD(learning_rate=0.0001)
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=["accuracy"])
#model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [11]:
from keras.callbacks import ModelCheckpoint
# define checkpoint callback
filepath = 'saved_models/model-ep{epoch:02d}-loss{loss:.3f}-val_loss{val_loss:.3f}.h5'
checkpoint = ModelCheckpoint(filepath, monitor='val_loss', verbose=1, save_best_only=False, mode='min')

In [12]:
#enter the weights of the model from where you start resuming incase the training is not going to be completed in one pass
#model.load_weights('C:\Users\Shubham\Downloads\Detection-of-Violent-Scenes-in-Cartoon-Movies-main\Detection-of-Violent-Scenes-in-Cartoon-Movies-main')

In [13]:
# import os
# import random
# import shutil

# train_dir = "Dataset/train_set"
# val_dir = "Dataset/val_set"

# # create validation folder if it does not exist
# os.makedirs(val_dir, exist_ok=True)

# classes = os.listdir(train_dir)

# for cls in classes:

#     class_train_path = os.path.join(train_dir, cls)
#     class_val_path = os.path.join(val_dir, cls)

#     os.makedirs(class_val_path, exist_ok=True)

#     files = os.listdir(class_train_path)

#     random.shuffle(files)

#     split_size = int(0.2 * len(files))   # 20% for validation

#     val_files = files[:split_size]

#     for file in val_files:
#         src = os.path.join(class_train_path, file)
#         dst = os.path.join(class_val_path, file)

#         shutil.move(src, dst)

#     print(f"{cls} -> moved {split_size} files to validation")

# print("Dataset split completed successfully!")

In [14]:
# import os

# root = "Dataset"

# for path, dirs, files in os.walk(root):
#     for file in files:
#         new_name = file.replace("(", "_").replace(")", "_").replace(" ", "_")

#         if new_name != file:
#             os.rename(
#                 os.path.join(path, file),
#                 os.path.join(path, new_name)
#             )

# print("Filename cleaning completed")

In [ ]:
batch_size = 8

train_dir = r"Dataset/train_set"
val_dir = r"Dataset/validation_set"

logdir = "logs/fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = keras.callbacks.TensorBoard(log_dir=logdir)

start_time = time.perf_counter()

history = model.fit(
    train_it,
    steps_per_epoch=213,
    epochs=100,
    validation_data=val_it,
    validation_steps=53,
    callbacks=[tensorboard_callback, checkpoint],
    verbose=1
)

print(time.perf_counter() - start_time)

# save model
model_json = model.to_json()
with open("model.json", "w") as json_file:
    json_file.write(model_json)

model.save_weights("best_model1.weights.h5")
print("Saved model to disk")

In [ ]:
# load json and create model
json_file = open('model.json', 'r')
loaded_model_json = json_file.read()
json_file.close()
loaded_model = model_from_json(loaded_model_json)
# load weights into new model
loaded_model.load_weights("best_model1.weights.h5")
print("Loaded model from disk")

In [ ]:
import tensorflow
opt = tensorflow.keras.optimizers.SGD(learning_rate=0.001)

loaded_model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=["accuracy"])

In [ ]:
import matplotlib.pyplot as plt

# list all data in history
print(history.history.keys())

# summarize history for accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val_test'], loc='upper left')
plt.show()


# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'Val_test'], loc='upper left')
plt.show()


In [ ]:
#evaluate the model
model.evaluate_generator(generator=val_it,
steps=300)

In [ ]:
#predict the output 

test_it.reset()
pred=model.predict_generator(test_it,
steps=324,
verbose=1)

In [ ]:
predicted_class_indices=np.argmax(pred,axis=1)